# Figure 5: DR2 BGS magnitude selection

This notebook performs one task: visualize the DR2 BGS Bright galaxy density in redshift and absolute-magnitude space, together with the apparent-magnitude selection boundary and the adopted $0.2L_*$ reference threshold.

In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.cosmology import Planck18
from astropy.io import fits
from matplotlib import colors
from matplotlib.colors import LinearSegmentedColormap, to_rgb

REPO_ROOT = Path('/global/homes/z/zzhang13/DESI/Projection')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'tools').exists():
        REPO_ROOT = REPO_ROOT.parent

BGS_DIR = Path('/global/cfs/cdirs/desi/survey/catalogs/DA2/LSS/loa-v1/LSScats/v2.1')
BGS_FILE = BGS_DIR / 'BGS_BRIGHT_full_noveto.dat.fits'
RANDOM_PATTERN = 'BGS_BRIGHT_*_full.ran.fits'
LF_SUMMARY = REPO_ROOT / 'catalogs' / 'bgs_direct_lf_logL_global_vmax_schechter_fit_summary.csv'
OUTPUT_FIGURE = REPO_ROOT / 'plots' / 'M_r_vs_z_Pngal.jpeg'

RANDOM_DENSITY_DEG2 = 2500.0
R_MAG_LIMIT = 19.5
M_SUN_R_AB = 4.64
Z_RANGE = (0.10, 0.40)
M_RANGE = (-26.0, -16.0)
DZ = 0.001
DM = 0.02

print('BGS catalog:', BGS_FILE)
print('LF summary:', LF_SUMMARY)
print('Output:', OUTPUT_FIGURE)

In [ ]:
if not BGS_FILE.exists():
    raise FileNotFoundError(BGS_FILE)
if not LF_SUMMARY.exists():
    raise FileNotFoundError(LF_SUMMARY)

with fits.open(BGS_FILE, memmap=True) as hdul:
    data = hdul[1].data
    z_gal = np.array(data['Z'], dtype=float, copy=True)
    flux_r = np.array(data['FLUX_R'], dtype=float, copy=True)
    prob_obs = np.array(data['PROB_OBS'], dtype=float, copy=True)

random_files = sorted(BGS_DIR.glob(RANDOM_PATTERN))
if not random_files:
    raise FileNotFoundError(f'No DR2 random catalogs matched {BGS_DIR / RANDOM_PATTERN}')
with fits.open(random_files[0], memmap=True) as hdul:
    n_random = int(hdul[1].header['NAXIS2'])
survey_area_deg2 = n_random / RANDOM_DENSITY_DEG2
f_sky = survey_area_deg2 / 41252.96124941927

cosmo = Planck18
h = cosmo.H0.value / 100.0
r_mag = np.full(len(flux_r), np.nan, dtype=float)
positive_flux = np.isfinite(flux_r) & (flux_r > 0)
r_mag[positive_flux] = 22.5 - 2.5 * np.log10(flux_r[positive_flux])
M_r_h = np.full(len(z_gal), np.nan, dtype=float)
valid_distance = positive_flux & np.isfinite(z_gal) & (z_gal > 0)
M_r_h[valid_distance] = (
    r_mag[valid_distance]
    - cosmo.distmod(z_gal[valid_distance]).value
    - 5.0 * np.log10(h)
)

comp_weight = np.full(len(prob_obs), np.nan, dtype=float)
good_prob = np.isfinite(prob_obs) & (prob_obs > 0)
comp_weight[good_prob] = 1.0 / prob_obs[good_prob]

lf_summary = pd.read_csv(LF_SUMMARY)
if len(lf_summary) == 0 or 'log10_L_star' not in lf_summary.columns:
    raise ValueError('LF summary must contain at least one log10_L_star value')
log10_L_star = float(lf_summary.loc[0, 'log10_L_star'])
log10_L_02star = log10_L_star + np.log10(0.2)
M_02Lstar = M_SUN_R_AB - 2.5 * log10_L_02star

print(f'DR2 random-derived area: {survey_area_deg2:,.1f} deg^2')
print(f'Using log10(L_star): {log10_L_star:.4f}')
print(f'0.2 L_star threshold: M_r - 5 log10(h) = {M_02Lstar:.3f}')

In [ ]:
z_min, z_max = Z_RANGE
M_min, M_max = M_RANGE
z_edges = np.arange(z_min, z_max + DZ, DZ)
M_edges = np.arange(M_min, M_max + DM, DM)
z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])
M_centers = 0.5 * (M_edges[:-1] + M_edges[1:])

valid = (
    np.isfinite(z_gal)
    & np.isfinite(M_r_h)
    & np.isfinite(comp_weight)
    & (z_gal >= z_min)
    & (z_gal < z_max)
    & (M_r_h >= M_min)
    & (M_r_h < M_max)
)

counts, _, _ = np.histogram2d(
    z_gal[valid],
    M_r_h[valid],
    weights=comp_weight[valid],
    bins=[z_edges, M_edges],
)

distance_edges = cosmo.comoving_distance(z_edges).to_value(u.Mpc)
shell_volume_mpc3 = (4.0 * np.pi / 3.0) * f_sky * np.diff(distance_edges**3)
shell_volume_hmpc3 = shell_volume_mpc3 * h**3
density = counts / shell_volume_hmpc3[:, None] / DM

M_limit_curve = R_MAG_LIMIT - cosmo.distmod(z_centers).value - 5.0 * np.log10(h)
Z_grid, M_grid = np.meshgrid(z_centers, M_centers, indexing='ij')
M_limit_grid = R_MAG_LIMIT - cosmo.distmod(Z_grid).value - 5.0 * np.log10(h)
forbidden = M_grid > M_limit_grid
masked_density = np.ma.masked_where(
    (density <= 0) | ~np.isfinite(density) | forbidden,
    density,
)

positive = masked_density.compressed()
if len(positive) == 0:
    raise RuntimeError('No populated magnitude-redshift bins remain after selection')
vmin = np.nanpercentile(positive, 2)
vmax = np.nanpercentile(positive, 99.5)

cmap = LinearSegmentedColormap.from_list(
    'BGS_LF_density',
    [to_rgb('white'), to_rgb('thistle'), to_rgb('royalblue')],
    N=256,
)
cmap.set_bad('white')

fig, ax = plt.subplots(figsize=(6.5, 6.0))
mesh = ax.pcolormesh(
    z_edges,
    M_edges,
    masked_density.T,
    cmap=cmap,
    norm=colors.LogNorm(vmin=vmin, vmax=vmax),
    shading='auto',
)

cbar = fig.colorbar(mesh, ax=ax, orientation='vertical', location='right')
cbar.set_label(r'$n_{\rm gal}\;[h^3\,{\rm Mpc}^{-3}\,{\rm mag}^{-1}]$')
ax.axhline(M_02Lstar, color='C2', lw=3, ls='-.', label=r'$0.2L_*$')
ax.plot(z_centers, M_limit_curve, color='C3', lw=3, ls='--', label=rf'$m_r={R_MAG_LIMIT:.1f}$')

for z_mark in [0.18, 0.24, 0.35]:
    M_mark = R_MAG_LIMIT - cosmo.distmod(z_mark).value - 5.0 * np.log10(h)
    ax.plot([z_min, z_mark], [M_mark, M_mark], color='0.2', lw=1.5, ls=':')
    ax.plot([z_mark, z_mark], [M_min, M_mark], color='0.2', lw=1.5, ls=':')

ax.set_xlim(z_min, z_max)
ax.set_ylim(M_max, M_min)
ax.set_xlabel(r'$z$')
ax.set_ylabel(r'$M_r-5\log_{10}h$')
ax.legend(loc='lower left', frameon=False)
ax.tick_params(direction='in', top=True, right=True)
fig.tight_layout()
OUTPUT_FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_FIGURE, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {OUTPUT_FIGURE}')